In [1]:
!pip install scikit-learn pandas numpy

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [2]:
# read uploaded file
columns = ['user_id', 'item_id', 'rating', 'timestamp']

ratings = pd.read_csv('u.data',
                      sep='\t',
                      names=columns)

ratings.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [5]:
ratings.shape

(100000, 4)

In [6]:
ratings.describe()

,user_id,item_id,rating,timestamp
count,100000.00000,100000.000000,100000.000000,1.000000e+05
mean,462.48475,425.530130,3.529860,8.835289e+08
std,266.61442,330.798356,1.125674,5.343856e+06
min,1.00000,1.000000,1.000000,8.747247e+08
25%,254.00000,175.000000,3.000000,8.794487e+08
50%,447.00000,322.000000,4.000000,8.828269e+08
75%,682.00000,631.000000,4.000000,8.882600e+08
max,943.00000,1682.000000,5.000000,8.932866e+08


In [7]:
ratings.nunique()

,0
user_id,943
item_id,1682
rating,5
timestamp,49282


In [8]:
user_item_matrix = ratings.pivot(index='user_id',
                                  columns='item_id',
                                  values='rating')

user_item_matrix = user_item_matrix.fillna(0)

user_item_matrix.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

train_matrix = train.pivot(index='user_id',
                           columns='item_id',
                           values='rating').fillna(0)

test_matrix = test.pivot(index='user_id',
                         columns='item_id',
                         values='rating').fillna(0)

In [10]:
user_similarity = cosine_similarity(train_matrix)
user_similarity_df = pd.DataFrame(user_similarity,
                                  index=train_matrix.index,
                                  columns=train_matrix.index)

user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.136196,0.030424,0.026203,0.284613,0.331412,0.319056,0.274139,0.083486,0.281396,...,0.277459,0.084849,0.205849,0.144161,0.133679,0.092367,0.216948,0.084181,0.104599,0.329288
2,0.136196,1.000000,0.114644,0.168220,0.093128,0.162165,0.095848,0.091360,0.149476,0.125701,...,0.149359,0.268977,0.320095,0.323347,0.241012,0.152655,0.230951,0.117484,0.166632,0.096719
3,0.030424,0.114644,1.000000,0.346894,0.000000,0.085071,0.032829,0.053875,0.060177,0.052552,...,0.021713,0.017707,0.154299,0.049358,0.107604,0.019022,0.101207,0.021959,0.127179,0.013805
4,0.026203,0.168220,0.346894,1.000000,0.011848,0.051287,0.075209,0.142100,0.060465,0.035202,...,0.034908,0.044480,0.087428,0.118082,0.100612,0.000000,0.151086,0.110324,0.112342,0.032367
5,0.284613,0.093128,0.000000,0.011848,1.000000,0.168527,0.298438,0.185290,0.039737,0.166013,...,0.276012,0.103529,0.085547,0.072429,0.104445,0.049198,0.204472,0.148028,0.099978,0.247527


In [11]:
def predict_user_based(user_id, item_id):
    if item_id not in train_matrix.columns:
        return 0

    sim_users = user_similarity_df[user_id]
    item_ratings = train_matrix[item_id]

    numerator = np.dot(sim_users, item_ratings)
    denominator = np.sum(np.abs(sim_users))

    if denominator == 0:
        return 0

    return numerator / denominator

In [12]:
def get_recommendations_user(user_id, top_n=10):
    scores = []
    for item in train_matrix.columns:
        if train_matrix.loc[user_id, item] == 0:
            score = predict_user_based(user_id, item)
            scores.append((item, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [i[0] for i in scores[:top_n]]

In [13]:
def precision_recall_user(test, top_n=10):
    precisions = []
    recalls = []

    for user in test['user_id'].unique():
        if user not in train_matrix.index:
            continue

        recommended = get_recommendations_user(user, top_n)
        relevant = test[(test['user_id']==user) & (test['rating']>=4)]['item_id'].tolist()

        if len(relevant) == 0:
            continue

        tp = len(set(recommended) & set(relevant))
        precision = tp / top_n
        recall = tp / len(relevant)

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

In [14]:
precision_user, recall_user = precision_recall_user(test)
f1_user = 2 * (precision_user * recall_user) / (precision_user + recall_user)

precision_user, recall_user, f1_user

(np.float64(0.19793478260869568),
 np.float64(0.20489403812719847),
 np.float64(0.2013542964499765))

In [15]:
item_similarity = cosine_similarity(train_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity,
                                  index=train_matrix.columns,
                                  columns=train_matrix.columns)

item_similarity_df.head()

item_id,1,2,3,4,5,6,7,8,9,10,...,1668,1670,1671,1672,1673,1676,1678,1679,1680,1681
item_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.311393,0.253690,0.350312,0.214229,0.081377,0.520601,0.376806,0.395321,0.208464,...,0.0,0.0,0.0,0.000000,0.040426,0.000000,0.0,0.0,0.0,0.000000
2,0.311393,1.000000,0.216764,0.383544,0.304612,0.000000,0.297186,0.308952,0.222845,0.121722,...,0.0,0.0,0.0,0.062446,0.000000,0.000000,0.0,0.0,0.0,0.088312
3,0.253690,0.216764,1.000000,0.261066,0.141570,0.058716,0.271301,0.167238,0.224092,0.120622,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000
4,0.350312,0.383544,0.261066,1.000000,0.270286,0.061803,0.366117,0.370546,0.265128,0.190169,...,0.0,0.0,0.0,0.043872,0.000000,0.103406,0.0,0.0,0.0,0.062044
5,0.214229,0.304612,0.141570,0.270286,1.000000,0.017833,0.246926,0.196619,0.199997,0.036237,...,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000


In [16]:
def predict_item_based(user_id, item_id):
    if user_id not in train_matrix.index:
        return 0

    sim_items = item_similarity_df[item_id]
    user_ratings = train_matrix.loc[user_id]

    numerator = np.dot(sim_items, user_ratings)
    denominator = np.sum(np.abs(sim_items))

    if denominator == 0:
        return 0

    return numerator / denominator

In [17]:
def get_recommendations_item(user_id, top_n=10):
    scores = []
    for item in train_matrix.columns:
        if train_matrix.loc[user_id, item] == 0:
            score = predict_item_based(user_id, item)
            scores.append((item, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [i[0] for i in scores[:top_n]]

In [18]:
def precision_recall_item(test, top_n=10):
    precisions = []
    recalls = []

    for user in test['user_id'].unique():
        if user not in train_matrix.index:
            continue

        recommended = get_recommendations_item(user, top_n)
        relevant = test[(test['user_id']==user) & (test['rating']>=4)]['item_id'].tolist()

        if len(relevant) == 0:
            continue

        tp = len(set(recommended) & set(relevant))
        precision = tp / top_n
        recall = tp / len(relevant)

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

In [19]:
precision_item, recall_item = precision_recall_item(test)
f1_item = 2 * (precision_item * recall_item) / (precision_item + recall_item)

precision_item, recall_item, f1_item

(np.float64(0.0029347826086956524),
 np.float64(0.002623497478558359),
 np.float64(0.002770423459474761))

In [20]:
def predict_hybrid(user_id, item_id):
    user_score = predict_user_based(user_id, item_id)
    item_score = predict_item_based(user_id, item_id)

    return (user_score + item_score) / 2

In [22]:
def get_recommendations_hybrid(user_id, top_n=10):
    scores = []
    for item in train_matrix.columns:
        if train_matrix.loc[user_id, item] == 0:
            score = predict_hybrid(user_id, item)
            scores.append((item, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return [i[0] for i in scores[:top_n]]

In [25]:
def precision_recall_hybrid(test, top_n=10):
    precisions = []
    recalls = []

    for user in test['user_id'].unique():
        if user not in train_matrix.index:
            continue

        recommended = get_recommendations_hybrid(user, top_n)
        relevant = test[(test['user_id']==user) & (test['rating']>=4)]['item_id'].tolist()

        if len(relevant) == 0:
            continue

        tp = len(set(recommended) & set(relevant))
        precision = tp / top_n
        recall = tp / len(relevant)

        precisions.append(precision)
        recalls.append(recall)

    return np.mean(precisions), np.mean(recalls)

In [26]:
precision_hybrid, recall_hybrid = precision_recall_hybrid(test)
f1_hybrid = 2 * (precision_hybrid * recall_hybrid) / (precision_hybrid + recall_hybrid)

precision_hybrid, recall_hybrid, f1_hybrid

(np.float64(0.20467391304347826),
 np.float64(0.21127687300504194),
 np.float64(0.2079229840834474))

In [27]:
results = pd.DataFrame({
    'Model': ['User-Based CF', 'Item-Based CF', 'Hybrid CF'],
    'Precision': [precision_user, precision_item, precision_hybrid],
    'Recall': [recall_user, recall_item, recall_hybrid],
    'F1-Score': [f1_user, f1_item, f1_hybrid]
})

display(results)

,Model,Precision,Recall,F1-Score
0,User-Based CF,0.197935,0.204894,0.201354
1,Item-Based CF,0.002935,0.002623,0.002770
2,Hybrid CF,0.204674,0.211277,0.207923
